In [15]:
from pathlib import Path
import numpy as np
import pandas as pd

data_dir = Path("../data")
files = sorted(data_dir.rglob("raw_*.csv"))

print(f"Found {len(files)} asset files\n")

series_list = []

for path in files:
    asset = path.stem.replace("raw_", "")

    df = pd.read_csv(path, usecols=["Date", "Close"])
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")

    s = df.set_index("Date")["Close"]

    n_negative = (s < 0).sum()
    if n_negative > 0:
        print(f"  [{asset}] {n_negative} negative price(s) detected — forward-filled")

    s = s.where(s > 0, np.nan).ffill()
    log_returns = np.log(s / s.shift(1)).rename(asset)

    series_list.append(log_returns)

print(f"\nLoaded {len(series_list)} assets")

merged = pd.concat(series_list, axis=1, sort=False)
print(f"Merged shape before dropna: {merged.shape}")
print(f"Total NaNs before dropna: {merged.isna().sum().sum()}")

merged = merged.dropna()
print(f"Merged shape after dropna:  {merged.shape}")

merged = merged.reset_index()
merged = merged.sort_values("Date").reset_index(drop=True)

print(f"\nDate range: {merged['Date'].min()} → {merged['Date'].max()}")
print(f"Assets: {[c for c in merged.columns if c != 'Date']}")

merged["Date"] = merged["Date"].dt.strftime("%Y-%m-%d")
merged.to_csv(data_dir / "all_assets_log_returns.csv", index=False)

print(f"\nSaved → {data_dir / 'all_assets_log_returns.csv'}")
print(f"Final shape: {merged.shape[0]} trading days × {merged.shape[1] - 1} assets (+1 Date column)")

merged.head()


Found 30 asset files


Loaded 30 assets
Merged shape before dropna: (3774, 30)
Total NaNs before dropna: 129
Merged shape after dropna:  (3762, 30)

Date range: 2011-01-05 00:00:00 → 2025-12-30 00:00:00
Assets: ['agg', 'bnd', 'emb', 'hyg', 'ief', 'lqd', 'mub', 'shy', 'tip', 'tlt', 'coffee', 'copper', 'corn', 'crude_oil', 'gold', 'natural_gas', 'platinum', 'silver', 'soybeans', 'wheat', 'aapl', 'amzn', 'jnj', 'jpm', 'msft', 'nflx', 'nvda', 'tsla', 'v', 'xom']

Saved → ../data/all_assets_log_returns.csv
Final shape: 3762 trading days × 30 assets (+1 Date column)


,Date,agg,bnd,emb,hyg,ief,lqd,mub,shy,tip,...,aapl,amzn,jnj,jpm,msft,nflx,nvda,tsla,v,xom
0,2011-01-05,-0.004838,-0.004374,-0.005746,0.002531,-0.010832,-0.007366,-0.004516,-0.001789,-0.003827,...,0.008146,0.012942,-0.000632,0.012154,-0.003209,-0.009084,0.073927,0.005981,0.020885,-0.002673
1,2011-01-06,0.000190,0.000876,-0.007746,0.001208,0.005163,0.001477,-0.002821,0.001073,0.002895,...,-0.000808,-0.008358,-0.001581,-0.004934,0.028865,-0.009728,0.129622,0.038389,0.014870,0.006405
2,2011-01-07,0.003227,0.003748,0.002059,-0.003849,0.006628,0.005063,0.000908,0.001430,0.003165,...,0.007136,-0.001993,-0.009697,-0.019066,-0.007663,0.007333,0.027553,0.012830,-0.002874,0.005439
3,2011-01-10,0.002650,0.002366,-0.002527,-0.000771,0.003405,0.001743,-0.001513,0.000477,0.002878,...,0.018658,-0.004376,-0.007053,-0.005515,-0.013376,0.046743,0.037535,0.007409,-0.015888,-0.006104
4,2011-01-11,-0.001987,-0.001992,0.002714,0.003851,-0.003937,-0.000458,-0.000202,-0.000596,0.000370,...,-0.002368,-0.001843,0.001768,0.004598,-0.003905,-0.006568,-0.015633,-0.053794,0.000974,0.007426
